# River discharge hazard assessment — Kazakhstan

Adapted from the CLIMAAX [Handbook](https://handbook.climaax.eu/) and
[FLOODS/04_River_discharge_analysis](https://github.com/CLIMAAX/FLOODS)
GitHub repository.

## Data Substitutions

| Original (CLIMAAX / Europe) | Kazakhstan replacement | Reason |
|---|---|---|
| E-HYPEcatch European hydrological model (~5–10 km sub-basin polygons) | ISIMIP3b CWatM global hydrological model (0.5° grid cells, ≈50 km) | E-HYPEcatch covers only Europe |
| CLIMAAX cloud zarr mirror (pre-subset to catchment ID) | Direct ISIMIP3b file download with single-cell extraction | Mirror is Europe-only |
| E-HYPEcatch sub-basin polygon → catchment ID | Nearest 0.5° grid cell to user lon/lat | No global catchment polygon equivalent at fine resolution |
| 8 catchment model realisations (M00–M07) | Single GHM (CWatM); ensemble spread from 5 GCMs only | CWatM is one of several ISIMIP3b GHMs — see Deviation 3 |
| EURO-CORDEX GCM-RCM (RCP2.6/4.5/8.5) | ISIMIP3b GCMs under ssp126/ssp370 | Consistent with all other Kazakhstan workflows |
| Pre-computed 10yr/50yr return period discharge (E-HYPEcatch product) | GEV fitted to annual maxima from raw daily timeseries | No pre-computed discharge return period product exists for ISIMIP3b GHM output — see Deviation 2 |

## Key methodological deviations

**Deviation 1 — Grid cell vs catchment:**
The original workflow selects a specific E-HYPEcatch sub-basin polygon (5–10 km)
and reads discharge aggregated to that exact catchment boundary.
Here we snap to the nearest 0.5° grid cell (~50 km side). For large Kazakhstan
rivers (Irtysh, Syr Darya, Ural, Ili) this is scientifically valid — the grid
cell discharge represents routed flow through that cell. For small tributaries
(catchment area < ~5,000 km²), results should be interpreted with caution.

**Deviation 2 — Return period computation:**
The original loads pre-computed 10yr/50yr return period values directly from
the E-HYPEcatch dataset. Here we fit a Generalised Extreme Value (GEV)
distribution to annual maxima from raw daily timeseries — the same method used
in this repository's Extreme Precipitation workflow. Results carry more
uncertainty than a centrally computed product; at least 30 years are needed
for stable GEV fits.

**Deviation 3 — Single GHM:**
The original combines 8 catchment model realisations (M00–M07) with the GCM
ensemble, giving two axes of uncertainty. Here only CWatM is used as the GHM —
all ensemble spread comes from the 5 GCMs. To add more GHMs (H08, LPJmL, mHM,
MPI-HM), use the same URL and filename pattern with a different GHM name.
Not all GHMs are available for all GCMs — check data.isimip.org first.

In [23]:
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from scipy.optimize import minimize
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

try:
    import pooch
except ImportError:
    raise ImportError("Run: conda install -c conda-forge pooch")


logging.getLogger('pooch').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

In [2]:

loc_name = 'Irtysh_Semey'
loc_lon  = 80.27    
loc_lat  = 50.41    

GCMs = [
    'GFDL-ESM4',
    # 'IPSL-CM6A-LR',
    # 'MPI-ESM1-2-HR',
    # 'MRI-ESM2-0',
    # 'UKESM1-0-LL',
]
GHM       = 'CWatM'               
SCENARIOS = ['ssp126', 'ssp370']  

HIST_START = 1971   
HIST_END   = 2010   

FUTURE_START = 2041
FUTURE_END   = 2070

DOWNLOAD_FULL_FUTURE = False

RETURN_PERIODS = [10, 50]   

In [3]:
workflow_dir = Path('.')          
data_dir     = workflow_dir / 'data'
plot_dir     = workflow_dir / f'plots_{loc_name}'
results_dir  = workflow_dir / f'results_{loc_name}'

for d in [data_dir, plot_dir, results_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Location   : {loc_name}  ({loc_lon}°E, {loc_lat}°N)')
print(f'GHM        : {GHM}')
print(f'GCMs       : {GCMs}')
print(f'Scenarios  : {SCENARIOS}')
print(f'Hist range : {HIST_START}–{HIST_END}')
print(f'Future     : {FUTURE_START}–{FUTURE_END}  (mid-century)')

Location   : Irtysh_Semey  (80.27°E, 50.41°N)
GHM        : CWatM
GCMs       : ['GFDL-ESM4']
Scenarios  : ['ssp126', 'ssp370']
Hist range : 1971–2010
Future     : 2041–2070  (mid-century)


## Part 0 — Data download: ISIMIP3b CWatM discharge

ISIMIP3b GHM output files are available at:
  https://files.isimip.org/ISIMIP3b/OutputData/water_global/{GHM}/{GCM}/{scenario}/

File naming convention:
  `{ghm}_{gcm}_w5e5_{scenario}_{soc}_default_dis_global_daily_{start}_{end}.nc`

where soc = `histsoc` for historical, `2015soc` for future SSP.
Files are global 0.5° grids (~720×360 cells).  Strategy: download each decade
file, immediately extract the single grid cell of interest, save as a tiny
local .nc file, then delete the global file (same pattern as ESA WorldCover
in this repository).

If any URL returns HTTP 404, check the real file listing at:
  https://data.isimip.org/
  Search: model=cwatm, variable=dis, simulation_round=ISIMIP3b
and update `build_isimip3b_url()` accordingly.

In [4]:
ISIMIP_DIS_BASE = 'https://files.isimip.org/ISIMIP3b/OutputData/water_global'

SOC_HIST        = 'histsoc'
SOC_FUTURE      = '2015soc-from-histsoc'

def build_dis_url(gcm: str, scenario: str, decade_start: int):
    gcm_lower   = gcm.lower()
    ghm_lower   = GHM.lower()
    decade_end  = decade_start + 9
    soc         = SOC_HIST if scenario == 'historical' else SOC_FUTURE
    scen_folder = 'historical' if scenario == 'historical' else 'future'
    fname = (
        f'{ghm_lower}_{gcm_lower}_w5e5_{scenario}_{soc}'
        f'_default_dis_global_daily_{decade_start}_{decade_end}.nc'
    )
    url = f'{ISIMIP_DIS_BASE}/{GHM}/{gcm_lower}/{scen_folder}/{fname}'
    return url, fname


def download_and_extract_point(gcm: str, scenario: str, decade_start: int,
                                lon: float, lat: float, dest: Path) -> Path:
    url, fname  = build_dis_url(gcm, scenario, decade_start)
    global_path = dest / fname
    point_fname = fname.replace('_global_daily_', '_point_daily_')
    point_path  = dest / point_fname

    if point_path.exists():
        print(f'  Cached: {point_fname}')
        return point_path

    if not global_path.exists():
        print(f'  Downloading: {fname}')
        pooch.retrieve(url=url, known_hash=None, path=dest, fname=fname)

    with xr.open_dataset(global_path) as ds:
        lat_nm = next((c for c in ('lat', 'latitude', 'y') if c in ds.coords), None)
        lon_nm = next((c for c in ('lon', 'longitude', 'x') if c in ds.coords), None)
        if lat_nm is None or lon_nm is None:
            raise KeyError(f'Cannot identify lat/lon coords. Available: {list(ds.coords)}')

        var_nm = next(
            (v for v in ('dis', 'discharge', 'streamflow', 'qtot') if v in ds.data_vars),
            None
        )
        if var_nm is None:
            raise KeyError(f'Cannot find discharge variable. Available: {list(ds.data_vars)}')

        ds_sorted = ds.sortby(lat_nm)   
        point = (ds_sorted[var_nm]
                 .sel({lat_nm: lat, lon_nm: lon}, method='nearest')
                 .load())
        point.name = 'dis'
        point.to_netcdf(point_path)

    try:
        global_path.unlink()
        print(f'  Deleted global file: {fname}')
    except PermissionError:
        print(f'  NOTE: Could not delete {fname} — file still open.')
        print(f'  Restart kernel and re-run this cell to free disk space.')

    print(f'  Saved point timeseries: {point_fname}')
    return point_path

In [5]:
print('=== Historical discharge download ===')
hist_decades = list(range(HIST_START, HIST_END + 1, 10))
print(f'Decades: {hist_decades}')

hist_paths = {}   
for gcm in GCMs:
    print(f'\nGCM: {gcm}')
    hist_paths[gcm] = []
    for dec in hist_decades:
        p = download_and_extract_point(gcm, 'historical', dec, loc_lon, loc_lat, data_dir)
        hist_paths[gcm].append(p)

print('\nHistorical download complete.')

=== Historical discharge download ===
Decades: [1971, 1981, 1991, 2001]

GCM: GFDL-ESM4
  Cached: cwatm_gfdl-esm4_w5e5_historical_histsoc_default_dis_point_daily_1971_1980.nc
  Cached: cwatm_gfdl-esm4_w5e5_historical_histsoc_default_dis_point_daily_1981_1990.nc
  Cached: cwatm_gfdl-esm4_w5e5_historical_histsoc_default_dis_point_daily_1991_2000.nc
  Cached: cwatm_gfdl-esm4_w5e5_historical_histsoc_default_dis_point_daily_2001_2010.nc

Historical download complete.


In [6]:
print(f'\n=== Future discharge download ({FUTURE_START}–{FUTURE_END}) ===')
ssp_decades_mid = list(range(FUTURE_START, FUTURE_END + 1, 10))
print(f'Decades: {ssp_decades_mid}')

ssp_paths_mid = {sc: {} for sc in SCENARIOS}
for scenario in SCENARIOS:
    print(f'\nScenario: {scenario}')
    for gcm in GCMs:
        print(f'  GCM: {gcm}')
        ssp_paths_mid[scenario][gcm] = []
        for dec in ssp_decades_mid:
            p = download_and_extract_point(gcm, scenario, dec, loc_lon, loc_lat, data_dir)
            ssp_paths_mid[scenario][gcm].append(p)

print('\nFuture (mid-century) download complete.')


=== Future discharge download (2041–2070) ===
Decades: [2041, 2051, 2061]

Scenario: ssp126
  GCM: GFDL-ESM4
  Cached: cwatm_gfdl-esm4_w5e5_ssp126_2015soc-from-histsoc_default_dis_point_daily_2041_2050.nc
  Cached: cwatm_gfdl-esm4_w5e5_ssp126_2015soc-from-histsoc_default_dis_point_daily_2051_2060.nc
  Cached: cwatm_gfdl-esm4_w5e5_ssp126_2015soc-from-histsoc_default_dis_point_daily_2061_2070.nc

Scenario: ssp370
  GCM: GFDL-ESM4
  Cached: cwatm_gfdl-esm4_w5e5_ssp370_2015soc-from-histsoc_default_dis_point_daily_2041_2050.nc
  Cached: cwatm_gfdl-esm4_w5e5_ssp370_2015soc-from-histsoc_default_dis_point_daily_2051_2060.nc
  Cached: cwatm_gfdl-esm4_w5e5_ssp370_2015soc-from-histsoc_default_dis_point_daily_2061_2070.nc

Future (mid-century) download complete.


In [7]:
ssp_paths_full = ssp_paths_mid   

if DOWNLOAD_FULL_FUTURE:
    print('\n=== Full future download (2011–2100) ===')
    full_decades = list(range(2011, 2101, 10))
    ssp_paths_full = {sc: {} for sc in SCENARIOS}
    for scenario in SCENARIOS:
        print(f'\nScenario: {scenario}')
        for gcm in GCMs:
            print(f'  GCM: {gcm}')
            ssp_paths_full[scenario][gcm] = []
            for dec in full_decades:
                p = download_and_extract_point(gcm, scenario, dec, loc_lon, loc_lat, data_dir)
                ssp_paths_full[scenario][gcm].append(p)
    print('\nFull future download complete.')

In [9]:
FILL_THRESH  = 1e10
PHYS_MAX_M3S = 5e5

def load_point_da(paths_dict: dict) -> xr.DataArray:
    gcm_arrays = []
    for gcm, paths in paths_dict.items():
        ds_list = [xr.open_dataset(p) for p in paths]
        cat = xr.concat([d['dis'] for d in ds_list], dim='time').sortby('time')
        cat = cat.where((cat > 0) & (cat < PHYS_MAX_M3S))
        vals = cat.values.flatten()
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0 and np.std(vals) < 1e-3 * np.mean(vals):
            print(f'  WARNING: {gcm} — discharge series is near-constant '
                  f'(mean={np.mean(vals):.1f}, std={np.std(vals):.3f}). '
                  f'This grid cell may not be on the river channel in CWatM.')
        cat = cat.assign_coords(gcm=gcm).expand_dims('gcm')
        gcm_arrays.append(cat)
    return xr.concat(gcm_arrays, dim='gcm')


da_hist = load_point_da(hist_paths)

da_ssp_mid = {sc: load_point_da(ssp_paths_mid[sc]) for sc in SCENARIOS}

if DOWNLOAD_FULL_FUTURE:
    da_ssp_full = {sc: load_point_da(ssp_paths_full[sc]) for sc in SCENARIOS}
else:
    da_ssp_full = da_ssp_mid  

lat_coord = float(da_hist.lat if 'lat' in da_hist.coords else da_hist.latitude)
lon_coord = float(da_hist.lon if 'lon' in da_hist.coords else da_hist.longitude)
print(f'\nGrid cell snapped to : {lat_coord:.2f}°N, {lon_coord:.2f}°E')
print(f'Offset from request  : '
      f'{abs(lat_coord - loc_lat)*111:.1f} km N-S, '
      f'{abs(lon_coord - loc_lon)*111*np.cos(np.radians(loc_lat)):.1f} km E-W')
print(f'Historical period    : {str(da_hist.time.min().values)[:10]} → '
      f'{str(da_hist.time.max().values)[:10]}')
print(f'Units                : {da_hist.attrs.get("units", "not specified — expected m³/s")}')
print(f'GCMs loaded          : {da_hist.gcm.values.tolist()}')


Grid cell snapped to : 50.25°N, 80.25°E
Offset from request  : 17.8 km N-S, 1.4 km E-W
Historical period    : 1971-01-01 → 2010-12-31
Units                : m3 s-1
GCMs loaded          : ['GFDL-ESM4']


## Part 1 — Daily discharge timeseries

Direct parallel to the original CLIMAAX workflow.  The 1991–2005 window is
used for display, matching the original's default period.

**Deviation 1 reminder:** Each line is one GCM driving CWatM at 0.5° resolution.
The original shows multiple catchment model realisations per GCM; the uncertainty
range shown here is narrower (GCM-only, no GHM ensemble).

In [10]:
da_plot = da_hist.sel(time=slice('1991', '2005'))
colorlist = px.colors.qualitative.Set1

fig = go.Figure()
for ii, gcm in enumerate(da_hist.gcm.values):
    fig.add_trace(go.Scatter(
        x=da_plot.time.values,
        y=da_plot.sel(gcm=gcm).values,
        mode='lines',
        name=gcm,
        line={'color': colorlist[ii % len(colorlist)]},
    ))

fig.update_yaxes(range=[0, float(da_plot.max()) * 1.05])
fig.update_layout(
    height=500, width=1100,
    title_text=(
        f'<b>Daily river discharge — {loc_name}</b> (1991–2005)<br>'
        f'Grid cell: {lat_coord:.2f}°N, {lon_coord:.2f}°E  |  GHM: {GHM}<br>'
        f'<i>Deviation 1: 0.5° grid cell (~50 km), not a 5–10 km catchment polygon</i>'
    ),
    yaxis_title='River discharge [m³/s]',
    xaxis_title='Date',
    legend_title='GCM',
    showlegend=True,
)
fig.show()
fig.write_image(plot_dir / f'{loc_name}_daily_timeseries_1991_2005.png')

## Part 2 — Flow-duration curve

Direct parallel to the original CLIMAAX workflow.  Using the full historical
period (1971–2010, 40 years) rather than just 1991–2005 for more robust
statistics — the original recommends at least 15 years.

The original workflow uses this curve to compare modelled vs GRDC observed
discharge.  If GRDC data for Kazakhstan stations is available (Irtysh, Ural,
Syr Darya gauges are registered), the GRDC validation logic from notebook 3
of the original CLIMAAX workflow is directly portable — just substitute the
GRDC station file path.

In [11]:
fig = go.Figure()
for ii, gcm in enumerate(da_hist.gcm.values):
    vals = da_hist.sel(gcm=gcm).values
    vals = vals[~np.isnan(vals)]
    sorted_desc = np.sort(vals)[::-1]
    exc_pct = np.arange(1, len(sorted_desc) + 1) / len(sorted_desc) * 100
    fig.add_trace(go.Scatter(
        x=exc_pct, y=sorted_desc,
        mode='lines',
        name=gcm,
        line={'color': colorlist[ii % len(colorlist)]},
    ))

fig.update_yaxes(range=[0, float(da_hist.max()) * 1.02])
fig.update_layout(
    height=500, width=1100,
    title_text=(
        f'<b>Flow-duration curve — {loc_name}</b> (1971–2010)<br>'
        f'Grid cell: {lat_coord:.2f}°N, {lon_coord:.2f}°E  |  GHM: {GHM}'
    ),
    yaxis_title='River discharge [m³/s]',
    xaxis_title='Exceedance [%]',
    legend_title='GCM',
    legend={'x': 0.75, 'y': 0.95},
)
fig.show()
fig.write_image(plot_dir / f'{loc_name}_flow_duration_curve.png')

## Part 3 — Seasonal variation (monthly means)

Direct parallel to the original CLIMAAX workflow Part 2.

The original plots four panels: historical (1971–2000), near-term (2011–2040),
mid-century (2041–2070), end-century (2071–2100).
If `DOWNLOAD_FULL_FUTURE = False`, only historical and mid-century are shown.
Set `DOWNLOAD_FULL_FUTURE = True` above to download all future decades and
reproduce the full 4-panel layout.

In [12]:
da_hist_ref = da_hist.sel(time=slice('1971', '2000'))
monthly_hist = da_hist_ref.groupby('time.month').mean('time')   # (gcm, month)

monthly_ssp_mid = {sc: da_ssp_mid[sc].groupby('time.month').mean('time')
                   for sc in SCENARIOS}

if DOWNLOAD_FULL_FUTURE:
    def monthly_for_window(da_dict, y_start, y_end):
        return {sc: da_dict[sc].sel(time=slice(str(y_start), str(y_end)))
                              .groupby('time.month').mean('time')
                for sc in SCENARIOS}
    monthly_near = monthly_for_window(da_ssp_full, 2011, 2040)
    monthly_end  = monthly_for_window(da_ssp_full, 2071, 2100)
else:
    monthly_near = None
    monthly_end  = None

In [13]:
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
months = np.arange(1, 13)
y_max  = max(float(monthly_hist.max()),
             max(float(monthly_ssp_mid[sc].max()) for sc in SCENARIOS)) * 1.08

dashlist = ['dot', 'dash']
sc_colors = {'ssp126': 'steelblue', 'ssp370': 'darkred'}
sc_labels  = {'ssp126': 'SSP1-2.6', 'ssp370': 'SSP3-7.0'}

n_panels = 4 if DOWNLOAD_FULL_FUTURE else 2
panel_info = [('Historical (1971–2000)', monthly_hist, None)]
if DOWNLOAD_FULL_FUTURE:
    panel_info += [
        ('Near-term (2011–2040)', None, monthly_near),
        ('Mid-century (2041–2070)', None, monthly_ssp_mid),
        ('End-century (2071–2100)', None, monthly_end),
    ]
else:
    panel_info += [('Mid-century (2041–2070)', None, monthly_ssp_mid)]

fig = make_subplots(
    rows=n_panels, cols=1,
    shared_xaxes=True,
    y_title='Monthly mean discharge [m³/s]',
    vertical_spacing=0.06,
    subplot_titles=[p[0] for p in panel_info],
)

for r, (title, m_hist, m_ssp) in enumerate(panel_info, start=1):
    if m_hist is not None:
        for ii, gcm in enumerate(GCMs):
            fig.add_trace(go.Scatter(
                x=months, y=m_hist.sel(gcm=gcm).values,
                mode='lines+markers',
                line={'color': colorlist[ii % len(colorlist)], 'dash': 'dot'},
                opacity=0.5, name=gcm, legendgroup=gcm,
                showlegend=(r == 1),
            ), row=r, col=1)
        fig.add_trace(go.Scatter(
            x=months, y=m_hist.median('gcm').values,
            mode='lines+markers',
            line={'color': 'black', 'width': 3},
            name='Historical median', legendgroup='hist_med',
            showlegend=(r == 1),
        ), row=r, col=1)
    else:
        fig.add_trace(go.Scatter(
            x=months, y=monthly_hist.median('gcm').values,
            mode='lines+markers', line={'color': 'lightgrey', 'width': 2},
            name='Historical median (ref)', legendgroup='hist_ref',
            showlegend=(r == 2),
        ), row=r, col=1)
        for ss, sc in enumerate(SCENARIOS):
            for ii, gcm in enumerate(GCMs):
                fig.add_trace(go.Scatter(
                    x=months, y=m_ssp[sc].sel(gcm=gcm).values,
                    mode='lines+markers',
                    line={'color': colorlist[ii % len(colorlist)],
                          'dash': dashlist[ss]},
                    opacity=0.4, name=gcm, legendgroup=gcm,
                    showlegend=False,
                ), row=r, col=1)
            fig.add_trace(go.Scatter(
                x=months, y=m_ssp[sc].median('gcm').values,
                mode='lines+markers',
                line={'color': sc_colors[sc], 'width': 3},
                name=f'{sc_labels[sc]} median', legendgroup=f'{sc}_med',
                showlegend=(r == 2),
            ), row=r, col=1)

fig.update_yaxes(range=[0, y_max])
fig.update_xaxes(
    tickvals=list(months), ticktext=month_labels,
    row=n_panels, col=1,
)
fig.update_layout(
    height=250 * n_panels, width=1100,
    title_text=(
        f'<b>Monthly mean river discharge — {loc_name}</b><br>'
        f'Grid cell: {lat_coord:.2f}°N, {lon_coord:.2f}°E  |  GHM: {GHM}  |  '
        f'Coloured lines = individual GCMs; bold lines = GCM median'
    ),
    showlegend=True,
    template='plotly_white',
    legend_title='GCM / scenario',
)
fig.show()
fig.write_image(plot_dir / f'{loc_name}_monthly_means_seasonal.png')

## Part 4 — Return period analysis (GEV fitted to annual maxima)

**Full explanation of Deviation 2:**

The original CLIMAAX workflow loads pre-computed 10yr and 50yr return period
discharge values directly from the E-HYPEcatch dataset for historical,
near-, mid-, and end-century periods under RCP scenarios.

Because no equivalent pre-computed product exists for ISIMIP3b GHM discharge,
we fit a Generalised Extreme Value (GEV) distribution to annual maxima
(block maxima method, block = calendar year).  This is the same method used
in the Extreme Precipitation workflow in this repository.

The GEV has three parameters: location μ, scale σ, and shape ξ.
scipy.stats.genextreme uses the sign convention c = −ξ.
A positive shape (ξ > 0, Fréchet) corresponds to a heavy-tailed distribution
typical of rivers with episodic flash flooding.

Historical record: 1971–2010 (40 annual maxima per GCM).
Future record: 2041–2070 (30 annual maxima per GCM).
Confidence intervals: 1000-sample bootstrap on GEV return levels.

**Limitation:** With 30–40 year records, the GEV shape parameter is uncertain.
Return level confidence intervals are shown; treat point estimates with caution
for return periods approaching or exceeding the record length.

In [24]:
ann_max_hist = da_hist.resample(time='YE').max() 

ann_max_ssp = {sc: da_ssp_mid[sc].resample(time='YE').max()
               for sc in SCENARIOS}

In [29]:
def fit_gev_return_levels(ann_max_1d: np.ndarray,
                           return_periods: list,
                           n_bootstrap: int = 1000,
                           shape_bounds: tuple = (-0.5, 0.5)):
    clean = ann_max_1d[~np.isnan(ann_max_1d)]
    clean = clean[(clean > 0) & (clean < PHYS_MAX_M3S)]
    out = {rp: {'point': np.nan, 'ci_low': np.nan, 'ci_high': np.nan}
           for rp in return_periods}
    if len(clean) < 10:
        return out

    def neg_loglik(params, data):
        c, loc, scale = params
        if scale <= 0:
            return 1e10
        ll = stats.genextreme.logpdf(data, c, loc=loc, scale=scale)
        return -np.sum(ll) if np.all(np.isfinite(ll)) else 1e10

    def fit_bounded(data):
        scale_g = float(np.std(data) * np.sqrt(6) / np.pi)
        loc_g   = float(np.mean(data) - 0.5772 * scale_g)

        try:
            c0, loc0, scale0 = stats.genextreme.fit(data)
            if abs(c0) > 2.0 or scale0 < 1e-3 * np.std(data):
                c0, loc0, scale0 = 0.0, loc_g, scale_g
        except Exception:
            c0, loc0, scale0 = 0.0, loc_g, scale_g
        c0 = float(np.clip(c0, *shape_bounds))
        result = minimize(
            neg_loglik,
            x0=[c0, loc0, scale0],
            args=(data,),
            bounds=[shape_bounds, (None, None), (1e-6, None)],
            method='L-BFGS-B',
        )
        return result.x if result.success else np.array([c0, loc0, scale0])

    c, loc, scale = fit_bounded(clean)
    print(f'    GEV fit: c={c:.4f}  loc={loc:.1f}  scale={scale:.1f}')

    boot_rls = {rp: [] for rp in return_periods}
    rng = np.random.default_rng(42)
    for _ in range(n_bootstrap):
        sample = rng.choice(clean, size=len(clean), replace=True)
        try:
            bc, bloc, bscale = fit_bounded(sample)
            for rp in return_periods:
                rl = stats.genextreme.ppf(1 - 1/rp, bc, loc=bloc, scale=bscale)
                if np.isfinite(rl) and 0 < rl < 1e7:
                    boot_rls[rp].append(rl)
        except Exception:
            pass

    for rp in return_periods:
        point = stats.genextreme.ppf(1 - 1/rp, c, loc=loc, scale=scale)
        bvals = np.array(boot_rls[rp])
        out[rp] = {
            'point':   float(point),
            'ci_low':  float(np.percentile(bvals, 5))  if len(bvals) else np.nan,
            'ci_high': float(np.percentile(bvals, 95)) if len(bvals) else np.nan,
        }
    return out


print('=== Historical GEV return levels (1971–2010) ===')
hist_rl = {}
for gcm in GCMs:
    vals = ann_max_hist.sel(gcm=gcm).values
    rl   = fit_gev_return_levels(vals, RETURN_PERIODS)
    hist_rl[gcm] = rl
    s = '  '.join(f'RP{rp}: {rl[rp]["point"]:.0f} m³/s' for rp in RETURN_PERIODS)
    print(f'  {gcm}: {s}')

print(f'\n=== Future GEV return levels ({FUTURE_START}–{FUTURE_END}) ===')
ssp_rl = {sc: {} for sc in SCENARIOS}
for sc in SCENARIOS:
    print(f'  {sc_labels[sc]}:')
    for gcm in GCMs:
        vals = ann_max_ssp[sc].sel(gcm=gcm).values
        rl   = fit_gev_return_levels(vals, RETURN_PERIODS)
        ssp_rl[sc][gcm] = rl
        s = '  '.join(f'RP{rp}: {rl[rp]["point"]:.0f} m³/s' for rp in RETURN_PERIODS)
        print(f'    {gcm}: {s}')

=== Historical GEV return levels (1971–2010) ===
    GEV fit: c=0.3447  loc=3755.2  scale=1207.5
  GFDL-ESM4: RP10: 5646 m³/s  RP50: 6346 m³/s

=== Future GEV return levels (2041–2070) ===
  SSP1-2.6:
    GEV fit: c=-0.0188  loc=3643.1  scale=968.2
    GFDL-ESM4: RP10: 5869 m³/s  RP50: 7563 m³/s
  SSP3-7.0:
    GEV fit: c=0.0363  loc=4040.1  scale=1391.3
    GFDL-ESM4: RP10: 7047 m³/s  RP50: 9102 m³/s


In [30]:
x_cat     = (['Historical\n(1971–2010)'] +
              [f'{sc_labels[sc]}\n({FUTURE_START}–{FUTURE_END})' for sc in SCENARIOS])
col_cat   = ['grey', 'steelblue', 'darkred']

fig = make_subplots(
    rows=1, cols=len(RETURN_PERIODS),
    subplot_titles=[f'{rp}-year return period discharge' for rp in RETURN_PERIODS],
)

for c_idx, rp in enumerate(RETURN_PERIODS):
    for g_idx, gcm in enumerate(GCMs):
        y_vals = [hist_rl[gcm][rp]['point']] + [ssp_rl[sc][gcm][rp]['point'] for sc in SCENARIOS]
        fig.add_trace(go.Scatter(
            x=x_cat, y=y_vals,
            mode='markers',
            marker={'color': colorlist[g_idx % len(colorlist)], 'size': 9},
            name=gcm, legendgroup=gcm,
            showlegend=(c_idx == 0),
        ), row=1, col=c_idx + 1)

    for x_idx, (x_lab, source) in enumerate(
        [('hist', hist_rl)] + [(sc, ssp_rl[sc]) for sc in SCENARIOS]
    ):
        median_val = np.nanmedian([source[g][rp]['point'] for g in GCMs])
        fig.add_trace(go.Scatter(
            x=[x_cat[x_idx]], y=[median_val],
            mode='markers',
            marker={'color': col_cat[x_idx], 'size': 14, 'symbol': 'diamond',
                    'line': {'color': 'black', 'width': 1}},
            name=f'GCM median ({x_lab})',
            showlegend=(c_idx == 0),
        ), row=1, col=c_idx + 1)

fig.update_yaxes(title='River discharge [m³/s]', rangemode='tozero')
fig.update_layout(
    height=500, width=900,
    title_text=(
        f'<b>Extreme river discharge return levels — {loc_name}</b><br>'
        f'GEV on annual maxima  |  GHM: {GHM}  |  '
        f'<i>Deviation 2: computed here vs pre-computed in original workflow</i>'
    ),
    showlegend=True,
    legend_title='GCM',
)
fig.show()
fig.write_image(plot_dir / f'{loc_name}_return_levels_absolute.png')

In [31]:
print(f'\n=== Relative change ({FUTURE_START}–{FUTURE_END} vs historical 1971–2010) ===')
fig = make_subplots(
    rows=1, cols=len(RETURN_PERIODS),
    subplot_titles=[f'{rp}-year: relative change [%]' for rp in RETURN_PERIODS],
)

for c_idx, rp in enumerate(RETURN_PERIODS):
    for sc in SCENARIOS:
        rel_vals = []
        for gcm in GCMs:
            h  = hist_rl[gcm][rp]['point']
            f  = ssp_rl[sc][gcm][rp]['point']
            rel_vals.append((f / h - 1) * 100 if (h and h > 0) else np.nan)

        sc_lbl = sc_labels[sc]
        print(f'  RP{rp} {sc_lbl}: {[f"{v:.1f}%" for v in rel_vals]} '
              f'→ median {np.nanmedian(rel_vals):.1f}%')

        fig.add_trace(go.Scatter(
            x=GCMs, y=rel_vals,
            mode='markers',
            marker={'color': sc_colors[sc], 'size': 10, 'opacity': 0.7},
            name=sc_lbl, legendgroup=sc,
            showlegend=(c_idx == 0),
        ), row=1, col=c_idx + 1)

        fig.add_trace(go.Scatter(
            x=['GCM median'], y=[np.nanmedian(rel_vals)],
            mode='markers',
            marker={'color': sc_colors[sc], 'size': 14, 'symbol': 'diamond',
                    'line': {'color': 'black', 'width': 1}},
            legendgroup=sc, showlegend=False,
        ), row=1, col=c_idx + 1)

fig.update_yaxes(title='Relative change [%]',
                 zeroline=True, zerolinecolor='black', zerolinewidth=1.5)
fig.update_layout(
    height=450, width=900,
    title_text=(
        f'<b>Relative change in extreme discharge — {loc_name}</b><br>'
        f'{FUTURE_START}–{FUTURE_END} vs historical 1971–2010  |  GEV on annual maxima'
    ),
    showlegend=True,
    legend_title='Scenario',
)
fig.show()
fig.write_image(plot_dir / f'{loc_name}_return_levels_relative_change.png')


=== Relative change (2041–2070 vs historical 1971–2010) ===
  RP10 SSP1-2.6: ['4.0%'] → median 4.0%
  RP10 SSP3-7.0: ['24.8%'] → median 24.8%
  RP50 SSP1-2.6: ['19.2%'] → median 19.2%
  RP50 SSP3-7.0: ['43.4%'] → median 43.4%


In [32]:
rows = []
for gcm in GCMs:
    for rp in RETURN_PERIODS:
        row = {
            'gcm': gcm,
            'return_period_yr': rp,
            'historical_1971_2010_m3s': hist_rl[gcm][rp]['point'],
            'historical_ci_low_m3s':   hist_rl[gcm][rp]['ci_low'],
            'historical_ci_high_m3s':  hist_rl[gcm][rp]['ci_high'],
        }
        for sc in SCENARIOS:
            f_pt = ssp_rl[sc][gcm][rp]['point']
            h_pt = hist_rl[gcm][rp]['point']
            row[f'{sc}_{FUTURE_START}_{FUTURE_END}_m3s']    = f_pt
            row[f'{sc}_{FUTURE_START}_{FUTURE_END}_ci_low']  = ssp_rl[sc][gcm][rp]['ci_low']
            row[f'{sc}_{FUTURE_START}_{FUTURE_END}_ci_high'] = ssp_rl[sc][gcm][rp]['ci_high']
            row[f'{sc}_pct_change'] = (f_pt / h_pt - 1) * 100 if (h_pt and h_pt > 0) else np.nan
        rows.append(row)

df_out = pd.DataFrame(rows)
csv_path = results_dir / f'{loc_name}_GEV_return_levels.csv'
df_out.to_csv(csv_path, index=False, float_format='%.2f')
print(f'\nSaved: {csv_path.name}')
print(df_out.to_string(index=False))


Saved: Irtysh_Semey_GEV_return_levels.csv
      gcm  return_period_yr  historical_1971_2010_m3s  historical_ci_low_m3s  historical_ci_high_m3s  ssp126_2041_2070_m3s  ssp126_2041_2070_ci_low  ssp126_2041_2070_ci_high  ssp126_pct_change  ssp370_2041_2070_m3s  ssp370_2041_2070_ci_low  ssp370_2041_2070_ci_high  ssp370_pct_change
GFDL-ESM4                10               5645.505973            5264.348057             5970.344184           5868.823458              5068.938344               6518.815813           3.955668           7046.631174              6249.882726               7961.196070          24.818417
GFDL-ESM4                50               6345.519142            5847.841199             6669.311316           7563.393356              5846.215207               9471.763166          19.192665           9102.197863              7763.678203              11263.011049          43.442919


## Conclusions

This workflow assessed river discharge hazard for a Kazakhstan river location
using ISIMIP3b CWatM global hydrological model output.

**Summary of outputs:**
- Daily timeseries (1991–2005): qualitative check on modelled discharge regime
- Flow-duration curve (1971–2010): basis for GRDC validation if gauge data available
- Seasonal variation: projected shifts in monthly discharge under SSP1-2.6 / SSP3-7.0
- GEV return levels: 10yr and 50yr discharge for historical and mid-century (2041–2070)

**Kazakhstan-specific notes:**
- Irtysh (Ertis): fed by Altai snowmelt; seasonal peak expected May–June.
- Syr Darya: heavily regulated for irrigation; CWatM output reflects the natural
  (pre-diversion) regime and should not be directly compared to observed gauged flows.
- Ural River: shared with Russia; upstream abstractions may affect downstream discharge.
- For GRDC validation: Kazakhstan has GRDC-registered stations on the Irtysh (Semey,
  Pavlodar), Ural (Oral), and Syr Darya (Kyzylorda). The GRDC validation logic from
  notebook 3 of the original CLIMAAX workflow is directly portable — substitute the
  GRDC station file path and catchment ID lookup with the lon/lat snap used here.

**Known limitations:**
- Single GHM (CWatM): no within-GHM ensemble uncertainty.
- 0.5° grid (~50 km): unsuitable for tributaries with catchment area < ~5,000 km².
- GEV fitted to 30–40 annual maxima: shape parameter uncertainty is substantial.
- Future analysis limited to mid-century (2041–2070); set DOWNLOAD_FULL_FUTURE=True
  to reproduce the original's near/mid/end-century four-panel seasonal plot.